# UA-DETRAC: inspect and prepare from local files or Colab.



In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import json

# Colab is optional; local runs find the project from the current folder.
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False

project_override = os.environ.get('UA_DETRAC_PROJECT')
candidates = [Path.cwd(), *Path.cwd().parents]
if project_override:
    candidates.insert(0, Path(project_override))
if IN_COLAB and not project_override:
    candidates.insert(0, Path('/content/drive/MyDrive/CS331/CS331-Vehicle-Tracking'))
ROOT_PATH = next(
    (path for path in candidates if (path / 'scripts/prepare_dataset.py').is_file()),
    None,
)
if ROOT_PATH is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')

sys.path.insert(0, str(ROOT_PATH))
WORK_ROOT = Path('/content/ua_detrac') if IN_COLAB else ROOT_PATH / 'outputs/ua_detrac'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_RAW_PATH = Path(os.environ.get('UA_DETRAC_RAW', ROOT_PATH / 'data/raw'))
if IN_COLAB and 'UA_DETRAC_RAW' not in os.environ:
    downloaded_raw = Path('/content/ua_detrac/raw')
    if downloaded_raw.is_dir():
        LOCAL_RAW_PATH = downloaded_raw
DATA_RAW_PATH = LOCAL_RAW_PATH
DATASET_HANDLE = 'bratjay/ua-detrac-orig/versions/2'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(ROOT_PATH / 'requirements.txt')],
    check=True,
)
print('Project:', ROOT_PATH)
print('Raw source:', DATA_RAW_PATH)
print('Work folder:', WORK_ROOT)
if IN_COLAB:
    print('Free space:', shutil.disk_usage(WORK_ROOT).free)









## Optional Kaggle download. Keep false when the raw dataset is already available locally.



In [ ]:
DOWNLOAD_DATASET = False
FORCE_DOWNLOAD = False
if DOWNLOAD_DATASET:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-U', 'kagglehub'],
        check=True,
    )
    import kagglehub

    download_path = WORK_ROOT / 'raw_download'
    DATA_RAW_PATH = Path(kagglehub.dataset_download(
        DATASET_HANDLE,
        output_dir=str(download_path),
        force_download=FORCE_DOWNLOAD,
    ))
    print('Downloaded from:', DATASET_HANDLE)
    print('Raw path:', DATA_RAW_PATH)
print('Raw top-level:', [path.name for path in DATA_RAW_PATH.iterdir()])



## Inspect files and annotation coverage before preparation. XML frame gaps are reported separately from missing files.



In [ ]:
LOCAL_INVENTORY = WORK_ROOT / 'raw_inventory.json'
result = subprocess.run(
    [sys.executable, str(ROOT_PATH / 'scripts/inspect_dataset.py'),
     '--raw', str(DATA_RAW_PATH), '--report', str(LOCAL_INVENTORY)],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError('Raw inventory failed; review the error above.')

inventory = json.loads(LOCAL_INVENTORY.read_text(encoding='utf-8'))
INVENTORY_COMPLETE = (
    inventory['ready_sequences'] == inventory['expected_sequences']
    and not inventory['unexpected_sequences']
)
print('All expected image/XML files structurally valid:', INVENTORY_COMPLETE)
print('Sequences with partial XML annotation:', inventory['partial_annotation_sequences'])



## Optional: compare with an earlier inventory saved in the project folder.



In [ ]:
previous = ROOT_PATH / 'data/raw_inventory.json'
if previous.is_file():
    old = json.loads(previous.read_text(encoding='utf-8'))
    old_images = {row['sequence'] for row in old['sequences'] if row['num_images'] > 0}
    local_images = {row['sequence'] for row in inventory['sequences'] if row['num_images'] > 0}
    print('Old inventory image sequences:', len(old_images))
    print('Current image sequences:', len(local_images))
    print('Only in current raw:', sorted(local_images - old_images))
else:
    print('No previous inventory found.')



## Next: run 00b in the same runtime. On local, output is under outputs/ua_detrac; on Colab it is under /content/ua_detrac.



## Optional: archive a verified raw backup. Raw remains unchanged; the inventory includes annotation coverage.



In [ ]:
BACKUP_RAW = False
if BACKUP_RAW:
    if not INVENTORY_COMPLETE:
        raise RuntimeError('Resolve structurally invalid files before calling this a complete raw backup.')
    subprocess.run(
        [sys.executable, str(ROOT_PATH / 'scripts/archive_raw.py'),
         '--raw', str(DATA_RAW_PATH),
         '--archive', str(WORK_ROOT / 'ua_detrac_raw.tar'),
         '--destination', str(ROOT_PATH / 'data/artifacts/raw_backup')],
        check=True,
    )

